# Generated

In [6]:
import pygame
import numpy as np
import random
from pygame.locals import *

# Constants
GRID_SIZE = 6
CELL_SIZE = 80
MARGIN = 50
PANEL_WIDTH = 200
WINDOW_WIDTH = GRID_SIZE * CELL_SIZE + MARGIN * 2 + PANEL_WIDTH
WINDOW_HEIGHT = GRID_SIZE * CELL_SIZE + MARGIN * 2

# Colors
BACKGROUND = (30, 30, 50)
WALL_COLOR = (50, 50, 70)
PLAYER_COLOR = (70, 130, 180)
GOAL_COLOR = (50, 180, 80)
HAZARD_COLOR = (220, 80, 60)
TEXT_COLOR = (220, 220, 220)
PANEL_COLOR = (40, 40, 60)
BUTTON_COLOR = (70, 130, 200)
ACTION_COLOR = (180, 180, 100)
VISITED_COLOR = (80, 80, 120)

# Rewards
GOAL_REWARD = 100
HAZARD_PENALTY = -50
STEP_PENALTY = -1

# 10 different maze layouts (0=path, 1=wall, 2=hazard, 3=goal)
MAZE_LAYOUTS = [
    # Level 1
    [
        [0, 0, 1, 0, 0, 0],
        [1, 0, 1, 0, 2, 0],
        [0, 0, 0, 0, 1, 3],
        [0, 1, 1, 0, 1, 0],
        [0, 2, 0, 0, 0, 0],
        [0, 1, 0, 1, 0, 0]
    ],
    # Level 2
    [
        [0, 1, 0, 0, 0, 0],
        [0, 1, 0, 1, 2, 0],
        [0, 0, 0, 1, 0, 0],
        [1, 0, 1, 1, 0, 1],
        [0, 0, 0, 0, 0, 3],
        [0, 1, 1, 0, 2, 0]
    ],
    # Level 3
    [
        [0, 1, 0, 0, 0, 1],
        [0, 0, 0, 1, 0, 0],
        [1, 1, 0, 0, 0, 1],
        [0, 0, 0, 1, 0, 0],
        [2, 0, 1, 0, 0, 3],
        [0, 0, 1, 0, 2, 0]
    ],
    # Level 4
    [
        [0, 0, 1, 1, 0, 0],
        [1, 0, 0, 0, 0, 1],
        [0, 0, 1, 0, 1, 0],
        [0, 1, 0, 0, 0, 0],
        [0, 0, 0, 1, 2, 3],
        [1, 0, 1, 0, 0, 0]
    ],
    # Level 5
    [
        [0, 1, 0, 0, 0, 0],
        [0, 1, 0, 1, 1, 0],
        [0, 0, 0, 0, 0, 0],
        [1, 0, 1, 1, 0, 1],
        [2, 0, 0, 0, 0, 3],
        [0, 1, 1, 0, 2, 0]
    ],
    # Level 6
    [
        [0, 0, 1, 0, 0, 0],
        [1, 0, 0, 0, 1, 0],
        [0, 0, 1, 0, 0, 0],
        [0, 1, 0, 0, 1, 0],
        [0, 0, 0, 1, 0, 3],
        [2, 0, 1, 0, 2, 0]
    ],
    # Level 7
    [
        [0, 1, 0, 0, 0, 0],
        [0, 0, 0, 1, 2, 0],
        [1, 0, 1, 0, 0, 0],
        [0, 0, 1, 0, 1, 0],
        [0, 1, 0, 0, 0, 3],
        [0, 0, 0, 1, 0, 0]
    ],
    # Level 8
    [
        [0, 0, 0, 1, 0, 0],
        [1, 1, 0, 0, 0, 1],
        [0, 0, 1, 0, 1, 0],
        [0, 1, 0, 0, 0, 0],
        [0, 0, 0, 1, 2, 3],
        [1, 0, 1, 0, 0, 0]
    ],
    # Level 9
    [
        [0, 1, 0, 0, 0, 0],
        [0, 0, 0, 1, 2, 0],
        [1, 0, 1, 0, 0, 0],
        [0, 0, 1, 1, 0, 1],
        [0, 1, 0, 0, 0, 3],
        [0, 0, 0, 1, 0, 0]
    ],
    # Level 10
    [
        [0, 0, 1, 0, 0, 0],
        [1, 0, 0, 0, 1, 0],
        [0, 0, 1, 1, 0, 0],
        [0, 1, 0, 0, 0, 1],
        [0, 0, 0, 1, 0, 3],
        [2, 0, 1, 0, 2, 0]
    ]
]

class MazeGame:
    def __init__(self):
        pygame.init()
        self.screen = pygame.display.set_mode((WINDOW_WIDTH, WINDOW_HEIGHT))
        pygame.display.set_caption("10-Level Maze RL - Visual Training")
        self.clock = pygame.time.Clock()
        self.font = pygame.font.SysFont('Arial', 16)
        self.big_font = pygame.font.SysFont('Arial', 24)
        
        self.current_level = 0
        self.grid = np.array(MAZE_LAYOUTS[self.current_level])
        self.player_pos = [0, 0]
        self.goal_pos = self.find_goal()
        self.mode = "manual"  # or "training"
        self.running = True
        self.episode_count = 0
        self.max_episodes = 10000
        self.training_complete = False
        self.success_count = 0
        self.level_completion_stats = [0] * 10  # Track deaths per level before completion
        self.visited_positions = set()  # Track visited positions
        self.last_action = None
        self.last_reward = 0
        self.paused = False
        
        # Q-learning parameters
        self.q_table = np.zeros((GRID_SIZE, GRID_SIZE, 4))  # (x, y, action)
        self.learning_rate = 0.1
        self.discount_factor = 0.9
        self.epsilon = 0.9
        self.min_epsilon = 0.1
        
        # UI buttons
        self.train_btn = pygame.Rect(GRID_SIZE*CELL_SIZE + MARGIN + 20, MARGIN + 50, 160, 30)
        self.manual_btn = pygame.Rect(GRID_SIZE*CELL_SIZE + MARGIN + 20, MARGIN + 90, 160, 30)
        self.reset_btn = pygame.Rect(GRID_SIZE*CELL_SIZE + MARGIN + 20, MARGIN + 130, 160, 30)
        self.pause_btn = pygame.Rect(GRID_SIZE*CELL_SIZE + MARGIN + 20, MARGIN + 170, 160, 30)
    
    def find_goal(self):
        for x in range(GRID_SIZE):
            for y in range(GRID_SIZE):
                if self.grid[x, y] == 3:
                    return [x, y]
        return [0, 0]  # Fallback
    
    def reset(self):
        self.player_pos = [0, 0]
        self.visited_positions = set()
        self.visited_positions.add((0, 0))
        self.last_action = None
        self.last_reward = 0
        return tuple(self.player_pos)
    
    def load_level(self, level):
        self.current_level = level
        self.grid = np.array(MAZE_LAYOUTS[self.current_level])
        self.goal_pos = self.find_goal()
        self.reset()
    
    def step(self, action):
        x, y = self.player_pos
        new_x, new_y = x, y
        self.last_action = action
        
        # 0=up, 1=right, 2=down, 3=left
        if action == 0 and x > 0: new_x = x - 1
        elif action == 1 and y < GRID_SIZE-1: new_y = y + 1
        elif action == 2 and x < GRID_SIZE-1: new_x = x + 1
        elif action == 3 and y > 0: new_y = y - 1
        
        # Check if move is valid (not a wall)
        if self.grid[new_x, new_y] != 1:
            self.player_pos = [new_x, new_y]
            self.visited_positions.add((new_x, new_y))
        
        # Check if player reached goal or hazard
        cell_value = self.grid[self.player_pos[0], self.player_pos[1]]
        if cell_value == 3:  # Goal
            reward = GOAL_REWARD
            done = True
            self.success_count += 1
            
            # Move to next level or complete training if finished all levels
            if self.current_level < 9:
                self.load_level(self.current_level + 1)
                done = False  # Continue training
            else:
                self.training_complete = True
                print("Training complete! Level completion stats:")
                for i, deaths in enumerate(self.level_completion_stats):
                    print(f"Level {i+1}: {deaths} deaths before completion")
                
        elif cell_value == 2:  # Hazard
            reward = HAZARD_PENALTY
            done = True
            # Record death for current level
            self.level_completion_stats[self.current_level] += 1
            # Reset to level 1
            self.load_level(0)
        else:  # Path
            reward = STEP_PENALTY
            done = False
        
        self.last_reward = reward
        
        if done:
            self.reset()
            self.episode_count += 1
            
            if self.episode_count >= self.max_episodes and not self.training_complete:
                self.training_complete = True
        
        return tuple(self.player_pos), reward, done
    
    def choose_action(self, state):
        if random.random() < self.epsilon:
            return random.randint(0, 3)  # Random action
        else:
            x, y = state
            return np.argmax(self.q_table[x, y])  # Best action
    
    def update_q_table(self, state, action, reward, next_state):
        x, y = state
        next_x, next_y = next_state
        best_next = np.max(self.q_table[next_x, next_y])
        
        # Q-learning formula
        self.q_table[x, y, action] += self.learning_rate * (
            reward + self.discount_factor * best_next - self.q_table[x, y, action]
        )
    
    def train_step(self):
        if not hasattr(self, 'current_state'):
            self.current_state = self.reset()
        
        action = self.choose_action(self.current_state)
        next_state, reward, done = self.step(action)
        self.update_q_table(self.current_state, action, reward, next_state)
        self.current_state = next_state
        
        # Reduce exploration over time
        self.epsilon = max(self.min_epsilon, self.epsilon * 0.9995)
        
        return done
    
    def handle_events(self):
        for event in pygame.event.get():
            if event.type == QUIT:
                self.running = False
            
            elif event.type == MOUSEBUTTONDOWN:
                if event.button == 1:  # Left click
                    mouse_pos = pygame.mouse.get_pos()
                    
                    if self.train_btn.collidepoint(mouse_pos):
                        self.mode = "training"
                        self.paused = False
                    elif self.manual_btn.collidepoint(mouse_pos):
                        self.mode = "manual"
                        self.paused = False
                    elif self.reset_btn.collidepoint(mouse_pos):
                        self.load_level(0)
                        self.episode_count = 0
                        self.success_count = 0
                        self.training_complete = False
                        self.level_completion_stats = [0] * 10
                        self.q_table = np.zeros((GRID_SIZE, GRID_SIZE, 4))
                        self.epsilon = 0.9
                        if hasattr(self, 'current_state'):
                            del self.current_state
                    elif self.pause_btn.collidepoint(mouse_pos) and self.mode == "training":
                        self.paused = not self.paused
            
            elif event.type == KEYDOWN and self.mode == "manual":
                if event.key == K_UP: self.step(0)
                elif event.key == K_RIGHT: self.step(1)
                elif event.key == K_DOWN: self.step(2)
                elif event.key == K_LEFT: self.step(3)
                elif event.key == K_SPACE and self.mode == "training":
                    self.paused = not self.paused
    
    def update(self):
        if self.mode == "training" and not self.training_complete and not self.paused:
            self.train_step()
    
    def draw(self):
        self.screen.fill(BACKGROUND)
        
        # Draw grid
        for x in range(GRID_SIZE):
            for y in range(GRID_SIZE):
                rect = pygame.Rect(
                    MARGIN + y * CELL_SIZE,
                    MARGIN + x * CELL_SIZE,
                    CELL_SIZE, CELL_SIZE
                )
                
                # Cell color - visited cells are darker
                if (x, y) in self.visited_positions:
                    pygame.draw.rect(self.screen, VISITED_COLOR, rect)
                
                if self.grid[x, y] == 1:  # Wall
                    pygame.draw.rect(self.screen, WALL_COLOR, rect)
                elif self.grid[x, y] == 2:  # Hazard
                    pygame.draw.rect(self.screen, HAZARD_COLOR, rect)
                elif self.grid[x, y] == 3:  # Goal
                    pygame.draw.rect(self.screen, GOAL_COLOR, rect)
                
                # Grid lines
                pygame.draw.rect(self.screen, (60, 60, 80), rect, 1)
                
                # Player
                if x == self.player_pos[0] and y == self.player_pos[1]:
                    pygame.draw.circle(
                        self.screen, PLAYER_COLOR,
                        (MARGIN + y * CELL_SIZE + CELL_SIZE//2, 
                         MARGIN + x * CELL_SIZE + CELL_SIZE//2),
                        CELL_SIZE//3
                    )
        
        # Draw action indicator
        if self.last_action is not None and self.mode == "training":
            x, y = self.player_pos
            arrow_pos = (MARGIN + y * CELL_SIZE + CELL_SIZE//2, 
                        MARGIN + x * CELL_SIZE + CELL_SIZE//2)
            
            # Draw arrow for last action
            if self.last_action == 0:  # Up
                pygame.draw.polygon(self.screen, ACTION_COLOR, [
                    (arrow_pos[0], arrow_pos[1] - 15),
                    (arrow_pos[0] - 10, arrow_pos[1] + 5),
                    (arrow_pos[0] + 10, arrow_pos[1] + 5)
                ])
            elif self.last_action == 1:  # Right
                pygame.draw.polygon(self.screen, ACTION_COLOR, [
                    (arrow_pos[0] + 15, arrow_pos[1]),
                    (arrow_pos[0] - 5, arrow_pos[1] - 10),
                    (arrow_pos[0] - 5, arrow_pos[1] + 10)
                ])
            elif self.last_action == 2:  # Down
                pygame.draw.polygon(self.screen, ACTION_COLOR, [
                    (arrow_pos[0], arrow_pos[1] + 15),
                    (arrow_pos[0] - 10, arrow_pos[1] - 5),
                    (arrow_pos[0] + 10, arrow_pos[1] - 5)
                ])
            elif self.last_action == 3:  # Left
                pygame.draw.polygon(self.screen, ACTION_COLOR, [
                    (arrow_pos[0] - 15, arrow_pos[1]),
                    (arrow_pos[0] + 5, arrow_pos[1] - 10),
                    (arrow_pos[0] + 5, arrow_pos[1] + 10)
                ])
        
        # Draw info panel
        panel = pygame.Rect(GRID_SIZE*CELL_SIZE + MARGIN, 0, PANEL_WIDTH, WINDOW_HEIGHT)
        pygame.draw.rect(self.screen, PANEL_COLOR, panel)
        
        # Draw buttons
        pygame.draw.rect(self.screen, BUTTON_COLOR, self.train_btn)
        pygame.draw.rect(self.screen, BUTTON_COLOR, self.manual_btn)
        pygame.draw.rect(self.screen, BUTTON_COLOR, self.reset_btn)
        pygame.draw.rect(self.screen, BUTTON_COLOR, self.pause_btn)
        
        # Draw button text
        train_text = self.font.render("Train Mode", True, TEXT_COLOR)
        manual_text = self.font.render("Manual Mode", True, TEXT_COLOR)
        reset_text = self.font.render("Reset Training", True, TEXT_COLOR)
        pause_text = self.font.render("Pause/Resume", True, TEXT_COLOR)
        
        self.screen.blit(train_text, (self.train_btn.x + 10, self.train_btn.y + 5))
        self.screen.blit(manual_text, (self.manual_btn.x + 10, self.manual_btn.y + 5))
        self.screen.blit(reset_text, (self.reset_btn.x + 10, self.reset_btn.y + 5))
        self.screen.blit(pause_text, (self.pause_btn.x + 10, self.pause_btn.y + 5))
        
        # Draw stats
        mode_text = self.font.render(f"Mode: {self.mode}", True, TEXT_COLOR)
        episode_text = self.font.render(f"Episodes: {self.episode_count}/{self.max_episodes}", True, TEXT_COLOR)
        level_text = self.font.render(f"Level: {self.current_level + 1}/10", True, TEXT_COLOR)
        epsilon_text = self.font.render(f"Exploration: {self.epsilon:.2f}", True, TEXT_COLOR)
        reward_text = self.font.render(f"Last reward: {self.last_reward}", True, TEXT_COLOR)
        action_text = self.font.render(f"Last action: {['Up', 'Right', 'Down', 'Left'][self.last_action] if self.last_action is not None else 'None'}", True, TEXT_COLOR)
        
        self.screen.blit(mode_text, (GRID_SIZE*CELL_SIZE + MARGIN + 20, MARGIN + 220))
        self.screen.blit(episode_text, (GRID_SIZE*CELL_SIZE + MARGIN + 20, MARGIN + 250))
        self.screen.blit(level_text, (GRID_SIZE*CELL_SIZE + MARGIN + 20, MARGIN + 280))
        self.screen.blit(epsilon_text, (GRID_SIZE*CELL_SIZE + MARGIN + 20, MARGIN + 310))
        self.screen.blit(reward_text, (GRID_SIZE*CELL_SIZE + MARGIN + 20, MARGIN + 340))
        self.screen.blit(action_text, (GRID_SIZE*CELL_SIZE + MARGIN + 20, MARGIN + 370))
        
        # Pause indicator
        if self.paused and self.mode == "training":
            pause_indicator = self.big_font.render("PAUSED", True, (220, 80, 60))
            self.screen.blit(pause_indicator, (GRID_SIZE*CELL_SIZE + MARGIN + 50, MARGIN + 400))
        
        # Training complete message
        if self.training_complete:
            complete_text = self.big_font.render("Training Complete!", True, (50, 180, 80))
            result_text = self.font.render(f"Level completion stats:", True, TEXT_COLOR)
            
            self.screen.blit(complete_text, (GRID_SIZE*CELL_SIZE + MARGIN + 20, MARGIN + 450))
            self.screen.blit(result_text, (GRID_SIZE*CELL_SIZE + MARGIN + 20, MARGIN + 480))
            
            # Display level completion stats
            for i in range(min(5, 10)):  # First 5 levels
                stat_text = self.font.render(
                    f"Level {i+1}: {self.level_completion_stats[i]} deaths", 
                    True, TEXT_COLOR
                )
                self.screen.blit(stat_text, (GRID_SIZE*CELL_SIZE + MARGIN + 20, MARGIN + 510 + i*30))
            
            if len(self.level_completion_stats) > 5:  # Next 5 levels if they exist
                for i in range(5, 10):
                    stat_text = self.font.render(
                        f"Level {i+1}: {self.level_completion_stats[i]} deaths", 
                        True, TEXT_COLOR
                    )
                    self.screen.blit(stat_text, (GRID_SIZE*CELL_SIZE + MARGIN + 120, MARGIN + 510 + (i-5)*30))
        
        pygame.display.flip()
    
    def run(self):
        while self.running:
            self.handle_events()
            self.update()
            self.draw()
            self.clock.tick(60)  # Limit to 60 FPS
        
        pygame.quit()

if __name__ == "__main__":
    game = MazeGame()
    game.run()

# langste volledige versie

In [8]:
import pygame
import numpy as np
import random
from pygame.locals import *

# Constants
GRID_SIZE = 6
CELL_SIZE = 80
MARGIN = 50
UI_HEIGHT = 100  # Verhoogd voor betere uitlijning
WINDOW_WIDTH = GRID_SIZE * CELL_SIZE * 2 + MARGIN * 3
WINDOW_HEIGHT = GRID_SIZE * CELL_SIZE + MARGIN * 2 + UI_HEIGHT

# Colors
BACKGROUND = (30, 30, 50)
WALL_COLOR = (50, 50, 70)
PLAYER_COLOR = (70, 130, 180)
GOAL_COLOR = (50, 180, 80)
HAZARD_COLOR = (220, 80, 60)
TEXT_COLOR = (220, 220, 220)
Q_VALUE_COLOR = (180, 180, 250)
ARROW_COLOR = (255, 255, 100)
EXPLORATION_COLOR = (250, 150, 50)

# Rewards
GOAL_REWARD = 100
HAZARD_PENALTY = -50
STEP_PENALTY = -1

# Maze layout (0=path, 1=wall, 2=hazard, 3=goal)
MAZE_LAYOUT = [
    [0, 0, 1, 0, 0, 0],
    [1, 0, 1, 0, 2, 0],
    [0, 0, 0, 0, 1, 3],
    [0, 1, 1, 0, 1, 0],
    [0, 2, 0, 0, 0, 0],
    [0, 1, 0, 1, 0, 0]
]

class MazeGame:
    def __init__(self):
        pygame.init()
        self.screen = pygame.display.set_mode((WINDOW_WIDTH, WINDOW_HEIGHT))
        pygame.display.set_caption("Maze Q-Learning")
        self.clock = pygame.time.Clock()
        self.font = pygame.font.SysFont('Arial', 16)
        self.font_small = pygame.font.SysFont('Arial', 14)
        
        self.grid = np.array(MAZE_LAYOUT)
        self.player_pos = [0, 0]
        self.goal_pos = self.find_goal()
        self.mode = "manual"  # "manual" or "training"
        self.running = True
        self.episode_count = 0
        
        # Q-learning parameters
        self.q_table = np.zeros((GRID_SIZE, GRID_SIZE, 4))  # (x, y, action)
        self.learning_rate = 0.1
        self.discount_factor = 0.9
        self.epsilon = 0.9
        self.min_epsilon = 0.1
        
        # Training parameters
        self.min_speed = 10   # 10 steps/sec
        self.max_speed = 1000  # 1,000 steps/sec
        self.steps_per_second = 100
        self.last_step_time = 0
        self.current_episode_state = None
        self.episode_done = True
        self.training_active = False
    
    def find_goal(self):
        for x in range(GRID_SIZE):
            for y in range(GRID_SIZE):
                if self.grid[x, y] == 3:
                    return [x, y]
        return [0, 0]  # Fallback
    
    def reset(self):
        self.player_pos = [0, 0]
        return tuple(self.player_pos)
    
    def step(self, action):
        x, y = self.player_pos
        new_x, new_y = x, y
        
        # 0=up, 1=right, 2=down, 3=left
        if action == 0 and x > 0: new_x = x - 1
        elif action == 1 and y < GRID_SIZE-1: new_y = y + 1
        elif action == 2 and x < GRID_SIZE-1: new_x = x + 1
        elif action == 3 and y > 0: new_y = y - 1
        
        # Check if move is valid (not a wall)
        if self.grid[new_x, new_y] != 1:
            self.player_pos = [new_x, new_y]
        
        # Check if player reached goal or hazard
        cell_value = self.grid[self.player_pos[0], self.player_pos[1]]
        if cell_value == 3:  # Goal
            reward = GOAL_REWARD
            done = True
        elif cell_value == 2:  # Hazard
            reward = HAZARD_PENALTY
            done = True
        else:  # Path
            reward = STEP_PENALTY
            done = False
        
        return tuple(self.player_pos), reward, done
    
    def choose_action(self, state):
        if random.random() < self.epsilon:
            return random.randint(0, 3)  # Random action
        else:
            x, y = state
            return np.argmax(self.q_table[x, y])  # Best action
    
    def update_q_table(self, state, action, reward, next_state):
        x, y = state
        next_x, next_y = next_state
        best_next = np.max(self.q_table[next_x, next_y])
        
        # Q-learning formula
        self.q_table[x, y, action] += self.learning_rate * (
            reward + self.discount_factor * best_next - self.q_table[x, y, action]
        )
    
    def training_step(self):
        """Take one training step - either start new episode or continue current one"""
        current_time = pygame.time.get_ticks()
        
        if self.episode_done:
            # Start new episode
            self.current_episode_state = self.reset()
            self.episode_done = False
            self.last_step_time = current_time
            return
        
        # Calculate how many steps we should take based on the speed
        delay_ms = 1000 / self.steps_per_second
        elapsed = current_time - self.last_step_time
        steps_to_take = min(100, int(elapsed / delay_ms))  # Limit to 100 steps per frame
        
        if steps_to_take > 0:
            for _ in range(steps_to_take):
                action = self.choose_action(self.current_episode_state)
                next_state, reward, done = self.step(action)
                self.update_q_table(self.current_episode_state, action, reward, next_state)
                
                self.current_episode_state = next_state
                self.last_step_time = current_time
                
                if done:
                    self.episode_done = True
                    self.episode_count += 1
                    # Reduce exploration over time
                    self.epsilon = max(self.min_epsilon, self.epsilon * 0.999)
                    break
    
    def draw_arrow(self, surface, start_pos, direction, strength):
        """Draw an arrow indicating the best action direction"""
        if strength <= 0:
            return
            
        # Arrow size based on Q-value strength
        arrow_size = max(10, min(30, int(strength * 3)))
        
        # Direction vectors: 0=up, 1=right, 2=down, 3=left
        directions = [
            (0, -arrow_size),   # up
            (arrow_size, 0),    # right
            (0, arrow_size),    # down
            (-arrow_size, 0)    # left
        ]
        
        if direction < len(directions):
            dx, dy = directions[direction]
            end_pos = (start_pos[0] + dx, start_pos[1] + dy)
            
            # Draw arrow line
            pygame.draw.line(surface, ARROW_COLOR, start_pos, end_pos, 3)
            
            # Draw arrowhead
            if direction == 0:  # up
                points = [end_pos, (end_pos[0]-5, end_pos[1]+8), (end_pos[0]+5, end_pos[1]+8)]
            elif direction == 1:  # right
                points = [end_pos, (end_pos[0]-8, end_pos[1]-5), (end_pos[0]-8, end_pos[1]+5)]
            elif direction == 2:  # down
                points = [end_pos, (end_pos[0]-5, end_pos[1]-8), (end_pos[0]+5, end_pos[1]-8)]
            else:  # left
                points = [end_pos, (end_pos[0]+8, end_pos[1]-5), (end_pos[0]+8, end_pos[1]+5)]
            
            pygame.draw.polygon(surface, ARROW_COLOR, points)
    
    def handle_events(self):
        for event in pygame.event.get():
            if event.type == QUIT:
                self.running = False
            
            elif event.type == KEYDOWN:
                if event.key == K_ESCAPE:
                    self.running = False
                elif event.key == K_t:
                    self.mode = "training"
                    self.training_active = True
                    self.episode_done = True
                elif event.key == K_m:
                    self.mode = "manual"
                    self.training_active = False
                    self.reset()  # Reset position when switching to manual
                elif event.key == K_r:
                    self.reset()
                    self.episode_count = 0
                    self.q_table = np.zeros((GRID_SIZE, GRID_SIZE, 4))
                    self.epsilon = 0.9
                    self.episode_done = True
                    self.training_active = False
                elif event.key == K_PLUS or event.key == K_EQUALS:
                    # Exponential speed increase (25% faster per keypress)
                    self.steps_per_second = min(self.max_speed, self.steps_per_second * 1.25)
                elif event.key == K_MINUS:
                    # Exponential speed decrease (20% slower per keypress)
                    self.steps_per_second = max(self.min_speed, self.steps_per_second * 0.8)
                elif event.key == K_SPACE:
                    # Pause/resume training
                    if self.mode == "training":
                        self.training_active = not self.training_active
                
                if self.mode == "manual":
                    if event.key == K_UP: 
                        _, _, done = self.step(0)
                        if done: self.reset()
                    elif event.key == K_RIGHT: 
                        _, _, done = self.step(1)
                        if done: self.reset()
                    elif event.key == K_DOWN: 
                        _, _, done = self.step(2)
                        if done: self.reset()
                    elif event.key == K_LEFT: 
                        _, _, done = self.step(3)
                        if done: self.reset()
    
    def update(self):
        if self.mode == "training" and self.training_active:
            self.training_step()
    
    def draw(self):
        self.screen.fill(BACKGROUND)
        
        # Draw main grid (environment)
        for x in range(GRID_SIZE):
            for y in range(GRID_SIZE):
                rect = pygame.Rect(
                    MARGIN + y * CELL_SIZE,
                    MARGIN + x * CELL_SIZE,
                    CELL_SIZE, CELL_SIZE
                )
                
                # Cell color
                if self.grid[x, y] == 1:  # Wall
                    pygame.draw.rect(self.screen, WALL_COLOR, rect)
                elif self.grid[x, y] == 2:  # Hazard
                    pygame.draw.rect(self.screen, HAZARD_COLOR, rect)
                elif self.grid[x, y] == 3:  # Goal
                    pygame.draw.rect(self.screen, GOAL_COLOR, rect)
                
                # Grid lines
                pygame.draw.rect(self.screen, (60, 60, 80), rect, 1)
                
                # Player
                if x == self.player_pos[0] and y == self.player_pos[1]:
                    pygame.draw.circle(
                        self.screen, PLAYER_COLOR,
                        (MARGIN + y * CELL_SIZE + CELL_SIZE//2, 
                         MARGIN + x * CELL_SIZE + CELL_SIZE//2),
                        CELL_SIZE//3
                    )
        
        # Draw Q-value grid with direction arrows
        for x in range(GRID_SIZE):
            for y in range(GRID_SIZE):
                rect = pygame.Rect(
                    MARGIN * 2 + GRID_SIZE * CELL_SIZE + y * CELL_SIZE,
                    MARGIN + x * CELL_SIZE,
                    CELL_SIZE, CELL_SIZE
                )
                
                # Background
                pygame.draw.rect(self.screen, (40, 40, 60), rect)
                pygame.draw.rect(self.screen, (60, 60, 80), rect, 1)
                
                # Skip walls
                if self.grid[x, y] == 1:
                    continue
                
                # Get Q-values for this cell
                q_values = self.q_table[x, y]
                max_q = np.max(q_values)
                best_action = np.argmax(q_values)
                
                if max_q > 0:  # Only draw if we have positive values
                    # Draw arrow for best direction
                    center = (rect.centerx, rect.centery)
                    self.draw_arrow(self.screen, center, best_action, max_q / 10)
                    
                    # Draw Q-value text
                    q_text = self.font.render(f"{max_q:.1f}", True, Q_VALUE_COLOR)
                    self.screen.blit(q_text, (rect.x + 5, rect.y + rect.height - 20))
        
        # Draw UI panel - perfect uitlijning
        panel_y = MARGIN + GRID_SIZE * CELL_SIZE + 20
        panel_width = WINDOW_WIDTH - MARGIN * 2
        panel_height = UI_HEIGHT - 20
        panel_x = MARGIN
        
        # Achtergrond paneel
        pygame.draw.rect(self.screen, (40, 40, 60), (panel_x, panel_y, panel_width, panel_height))
        pygame.draw.rect(self.screen, (60, 60, 80), (panel_x, panel_y, panel_width, panel_height), 2)
        
        # Definieer kolom posities en breedte
        col1_x = panel_x + 15
        col2_x = panel_x + 200
        col3_x = panel_x + 400
        col4_x = panel_x + 600
        
        # Definieer rij posities
        row1_y = panel_y + 15
        row2_y = panel_y + 35
        row3_y = panel_y + 55
        
        # Rij 1: Basis informatie
        mode_text = self.font.render(f"Mode: {self.mode}", True, TEXT_COLOR)
        self.screen.blit(mode_text, (col1_x, row1_y))
        
        status_text = "RUNNING" if (self.mode == "training" and self.training_active) else "PAUSED" if self.mode == "training" else "OFF"
        status_color = (50, 255, 50) if (self.mode == "training" and self.training_active) else (255, 255, 50) if self.mode == "training" else TEXT_COLOR
        training_text = self.font.render(f"Training: {status_text}", True, status_color)
        self.screen.blit(training_text, (col2_x, row1_y))
        
        episodes_text = self.font.render(f"Episodes: {self.episode_count}", True, TEXT_COLOR)
        self.screen.blit(episodes_text, (col3_x, row1_y))
        
        # Rij 2: Performance metrics
        if self.steps_per_second >= 1000:
            speed_str = f"{self.steps_per_second/1000:.1f}k steps/sec"
        else:
            speed_str = f"{self.steps_per_second:.0f} steps/sec"
        speed_text = self.font.render(f"Speed: {speed_str}", True, TEXT_COLOR)
        self.screen.blit(speed_text, (col1_x, row2_y))
        
        exploration_text = self.font.render(f"Exploration: {self.epsilon*100:.2f}%", True, EXPLORATION_COLOR)
        self.screen.blit(exploration_text, (col2_x, row2_y))
        
        # Rij 3: Controls - gecentreerd over de hele breedte
        controls_text = "Controls: T=Train | M=Manual | R=Reset | Space=Pause | +/-=Speed"
        controls_surface = self.font_small.render(controls_text, True, TEXT_COLOR)
        controls_x = panel_x + (panel_width - controls_surface.get_width()) // 2
        self.screen.blit(controls_surface, (controls_x, row3_y))
        
        pygame.display.flip()
    
    def run(self):
        while self.running:
            self.handle_events()
            self.update()
            self.draw()
            self.clock.tick(60)
        
        pygame.quit()

if __name__ == "__main__":
    game = MazeGame()
    game.run()


# Goat Simpelste Versie


In [28]:
import pygame
import numpy as np
import random

# Constanten
GRID_SIZE = 6
CELL_SIZE = 60
MARGIN = 30
WINDOW_WIDTH = GRID_SIZE * CELL_SIZE * 2 + MARGIN * 3
WINDOW_HEIGHT = GRID_SIZE * CELL_SIZE + MARGIN * 2 + 60

# Kleuren
BACKGROUND = (30, 30, 50)
WALL_COLOR = (50, 50, 70)
PLAYER_COLOR = (70, 130, 180)
GOAL_COLOR = (50, 180, 80)
HAZARD_COLOR = (220, 80, 60)
TEXT_COLOR = (220, 220, 220)

# Maze (0=pad, 1=muur, 2=gevaar, 3=doel)
MAZE = [
    [0, 0, 1, 0, 0, 0],
    [1, 0, 1, 0, 2, 0],
    [0, 0, 0, 0, 1, 3],
    [0, 1, 1, 0, 1, 0],
    [0, 2, 0, 0, 0, 0],
    [0, 1, 0, 1, 0, 0]
]

class SimpleMaze:
    def __init__(self):
        pygame.init()
        self.screen = pygame.display.set_mode((WINDOW_WIDTH, WINDOW_HEIGHT))
        pygame.display.set_caption("Q-Learning Maze")
        self.font = pygame.font.SysFont('Arial', 14)
        
        self.grid = np.array(MAZE)
        self.player_pos = [0, 0]
        self.mode = "manual"  # "manual" of "training"
        self.episodes = 0
        
        # Q-learning
        self.q_table = np.zeros((GRID_SIZE, GRID_SIZE, 4))
        self.epsilon = 0.9
        self.learning_rate = 0.1
        self.discount = 0.9
        
    def reset(self):
        self.player_pos = [0, 0]
        return tuple(self.player_pos)
    
    def move(self, action):
        x, y = self.player_pos
        # 0=omhoog, 1=rechts, 2=omlaag, 3=links
        moves = [(-1, 0), (0, 1), (1, 0), (0, -1)]
        dx, dy = moves[action]
        new_x, new_y = x + dx, y + dy
        
        # Check grenzen en muren
        if (0 <= new_x < GRID_SIZE and 0 <= new_y < GRID_SIZE and 
            self.grid[new_x, new_y] != 1):
            self.player_pos = [new_x, new_y]
        
        # Beloning berekenen
        cell = self.grid[self.player_pos[0], self.player_pos[1]]
        if cell == 3: return 100, True    # Doel
        elif cell == 2: return -50, True  # Gevaar
        else: return -1, False            # Stap
    
    def choose_action(self, state):
        if random.random() < self.epsilon:
            return random.randint(0, 3)
        x, y = state
        return np.argmax(self.q_table[x, y])
    
    def update_q(self, state, action, reward, next_state):
        x, y = state
        next_x, next_y = next_state
        best_next = np.max(self.q_table[next_x, next_y])
        
        self.q_table[x, y, action] += self.learning_rate * (
            reward + self.discount * best_next - self.q_table[x, y, action]
        )
    
    def train_step(self):
        state = tuple(self.player_pos)
        action = self.choose_action(state)
        reward, done = self.move(action)
        next_state = tuple(self.player_pos)
        
        self.update_q(state, action, reward, next_state)
        
        if done:
            self.reset()
            self.episodes += 1
            self.epsilon = max(0.1, self.epsilon * 0.999)
    
    def handle_input(self):
        for event in pygame.event.get():
            if event.type == pygame.QUIT:
                return False
            
            if event.type == pygame.KEYDOWN:
                if event.key == pygame.K_t:
                    self.mode = "training"
                elif event.key == pygame.K_m:
                    self.mode = "manual"
                    self.reset()
                elif event.key == pygame.K_r:
                    self.reset()
                    self.q_table = np.zeros((GRID_SIZE, GRID_SIZE, 4))
                    self.epsilon = 0.9
                    self.episodes = 0
                
                # Handmatige besturing
                if self.mode == "manual":
                    moves = {pygame.K_UP: 0, pygame.K_RIGHT: 1, 
                            pygame.K_DOWN: 2, pygame.K_LEFT: 3}
                    if event.key in moves:
                        _, done = self.move(moves[event.key])
                        if done: self.reset()
        return True
    
    def draw(self):
        self.screen.fill(BACKGROUND)
        
        # Teken beide grids
        for grid_offset in [0, GRID_SIZE * CELL_SIZE + MARGIN]:
            for x in range(GRID_SIZE):
                for y in range(GRID_SIZE):
                    rect = pygame.Rect(
                        MARGIN + y * CELL_SIZE + grid_offset,
                        MARGIN + x * CELL_SIZE,
                        CELL_SIZE, CELL_SIZE
                    )
                    
                    # Linker grid: maze
                    if grid_offset == 0:
                        if self.grid[x, y] == 1: color = WALL_COLOR
                        elif self.grid[x, y] == 2: color = HAZARD_COLOR
                        elif self.grid[x, y] == 3: color = GOAL_COLOR
                        else: color = BACKGROUND
                        
                        pygame.draw.rect(self.screen, color, rect)
                        pygame.draw.rect(self.screen, (60, 60, 80), rect, 1)
                        
                        # Speler
                        if x == self.player_pos[0] and y == self.player_pos[1]:
                            pygame.draw.circle(self.screen, PLAYER_COLOR, 
                                             rect.center, CELL_SIZE//4)
                    
                    # Rechter grid: Q-waarden
                    else:
                        pygame.draw.rect(self.screen, (40, 40, 60), rect)
                        pygame.draw.rect(self.screen, (60, 60, 80), rect, 1)
                        
                        if self.grid[x, y] != 1:  # Geen muur
                            max_q = np.max(self.q_table[x, y])
                            if max_q > 0:
                                text = self.font.render(f"{max_q:.1f}", True, TEXT_COLOR)
                                self.screen.blit(text, (rect.x + 5, rect.y + 5))
        
        # UI info
        info_y = MARGIN + GRID_SIZE * CELL_SIZE + 10
        info = f"Mode: {self.mode} | Episodes: {self.episodes} | Exploration: {self.epsilon*100:.0f}% | T=Train M=Manual R=Reset"
        text = self.font.render(info, True, TEXT_COLOR)
        self.screen.blit(text, (MARGIN, info_y))
        
        pygame.display.flip()
    
    def run(self):
        clock = pygame.time.Clock()
        running = True
        
        while running:
            running = self.handle_input()
            
            if self.mode == "training":
                for _ in range(10):  # 10 training stappen per frame
                    self.train_step()
            
            self.draw()
            clock.tick(60)
        
        pygame.quit()

if __name__ == "__main__":
    game = SimpleMaze()
    game.run()


# Simpele versie met snelheid

In [2]:
import pygame
import numpy as np
import random

# Constanten
GRID_SIZE = 6
CELL_SIZE = 80
MARGIN = 30
WINDOW_WIDTH = GRID_SIZE * CELL_SIZE * 2 + MARGIN * 3
WINDOW_HEIGHT = GRID_SIZE * CELL_SIZE + MARGIN * 2 + 80

# Kleuren
BACKGROUND = (30, 30, 50)
WALL_COLOR = (55, 55, 75)
PLAYER_COLOR = (70, 130, 180)
GOAL_COLOR = (50, 180, 80)
HAZARD_COLOR = (220, 80, 60)
TEXT_COLOR = (220, 220, 220)
ARROW_COLOR = (255, 255, 100)

# Maze (0=pad, 1=muur, 2=gevaar, 3=doel)
MAZE = [
    [0, 0, 1, 0, 0, 0],
    [1, 0, 1, 0, 2, 0],
    [0, 0, 0, 0, 1, 3],
    [0, 1, 1, 0, 1, 0],
    [0, 2, 0, 0, 0, 0],
    [0, 1, 0, 1, 0, 0]
]

class SimpleMaze:
    def __init__(self):
        pygame.init()
        self.screen = pygame.display.set_mode((WINDOW_WIDTH, WINDOW_HEIGHT))
        pygame.display.set_caption("Q-Learning Maze")
        self.font = pygame.font.SysFont('Arial', 14)
        
        self.grid = np.array(MAZE)
        self.player_pos = [0, 0]
        self.mode = "manual"
        self.episodes = 0
        
        # Q-learning
        self.q_table = np.zeros((GRID_SIZE, GRID_SIZE, 4))
        self.epsilon = 0.9
        self.learning_rate = 0.1
        self.discount = 0.9
        
        # Training control 
        self.steps_per_second = 100
        self.min_speed = 10
        self.max_speed = 1000
        self.last_step_time = 0
        self.current_state = None
        self.episode_done = True
        
    def reset(self):
        self.player_pos = [0, 0]
        return tuple(self.player_pos)
    
    def move(self, action):
        x, y = self.player_pos
        moves = [(-1, 0), (0, 1), (1, 0), (0, -1)]
        dx, dy = moves[action]
        new_x, new_y = x + dx, y + dy
        
        if (0 <= new_x < GRID_SIZE and 0 <= new_y < GRID_SIZE and 
            self.grid[new_x, new_y] != 1):
            self.player_pos = [new_x, new_y]
        
        cell = self.grid[self.player_pos[0], self.player_pos[1]]
        if cell == 3: return 100, True
        elif cell == 2: return -50, True
        else: return -1, False
    
    def choose_action(self, state):
        if random.random() < self.epsilon:
            return random.randint(0, 3)
        x, y = state
        return np.argmax(self.q_table[x, y])
    
    def update_q(self, state, action, reward, next_state):
        x, y = state
        next_x, next_y = next_state
        best_next = np.max(self.q_table[next_x, next_y])
        
        self.q_table[x, y, action] += self.learning_rate * (
            reward + self.discount * best_next - self.q_table[x, y, action]
        )
    
    def training_step(self):
        """Nieuwe architectuur - batch processing zoals eerste code"""
        current_time = pygame.time.get_ticks()
        
        if self.episode_done:
            self.current_state = self.reset()
            self.episode_done = False
            self.last_step_time = current_time
            return
        
        # Bereken hoeveel stappen we moeten nemen 
        delay_ms = 1000 / self.steps_per_second
        elapsed = current_time - self.last_step_time
        steps_to_take = min(50, int(elapsed / delay_ms))  # Max 50 per frame
        
        if steps_to_take > 0:
            for _ in range(steps_to_take):
                action = self.choose_action(self.current_state)
                reward, done = self.move(action)
                next_state = tuple(self.player_pos)
                
                self.update_q(self.current_state, action, reward, next_state)
                self.current_state = next_state
                
                if done:
                    self.episode_done = True
                    self.episodes += 1
                    self.epsilon = max(0.1, self.epsilon * 0.999)
                    break
            
            self.last_step_time = current_time
    
    def draw_arrow(self, surface, center, direction, strength):
        if strength <= 0:
            return
            
        arrow_size = max(8, min(20, int(strength * 2)))
        directions = [(0, -arrow_size), (arrow_size, 0), (0, arrow_size), (-arrow_size, 0)]
        
        if direction < len(directions):
            dx, dy = directions[direction]
            end_pos = (center[0] + dx, center[1] + dy)
            pygame.draw.line(surface, ARROW_COLOR, center, end_pos, 2)
            
            # Pijlpunt
            if direction == 0:
                points = [end_pos, (end_pos[0]-4, end_pos[1]+6), (end_pos[0]+4, end_pos[1]+6)]
            elif direction == 1:
                points = [end_pos, (end_pos[0]-6, end_pos[1]-4), (end_pos[0]-6, end_pos[1]+4)]
            elif direction == 2:
                points = [end_pos, (end_pos[0]-4, end_pos[1]-6), (end_pos[0]+4, end_pos[1]-6)]
            else:
                points = [end_pos, (end_pos[0]+6, end_pos[1]-4), (end_pos[0]+6, end_pos[1]+4)]
            
            pygame.draw.polygon(surface, ARROW_COLOR, points)
    
    def handle_input(self):
        for event in pygame.event.get():
            if event.type == pygame.QUIT:
                return False
            
            if event.type == pygame.KEYDOWN:
                if event.key == pygame.K_t:
                    self.mode = "training"
                    self.episode_done = True  # Start fresh episode
                elif event.key == pygame.K_m:
                    self.mode = "manual"
                    self.reset()
                elif event.key == pygame.K_r:
                    self.reset()
                    self.q_table = np.zeros((GRID_SIZE, GRID_SIZE, 4))
                    self.epsilon = 0.9
                    self.episodes = 0
                    self.episode_done = True
                elif event.key == pygame.K_PLUS or event.key == pygame.K_EQUALS:
                    self.steps_per_second = min(self.max_speed, int(self.steps_per_second * 1.25))
                elif event.key == pygame.K_MINUS:
                    self.steps_per_second = max(self.min_speed, int(self.steps_per_second * 0.8))
                
                if self.mode == "manual":
                    moves = {pygame.K_UP: 0, pygame.K_RIGHT: 1, 
                            pygame.K_DOWN: 2, pygame.K_LEFT: 3}
                    if event.key in moves:
                        _, done = self.move(moves[event.key])
                        if done: self.reset()
        return True
    
    def draw(self):
        self.screen.fill(BACKGROUND)
        
        # Beide grids tekenen
        for grid_offset in [0, GRID_SIZE * CELL_SIZE + MARGIN]:
            for x in range(GRID_SIZE):
                for y in range(GRID_SIZE):
                    rect = pygame.Rect(
                        MARGIN + y * CELL_SIZE + grid_offset,
                        MARGIN + x * CELL_SIZE,
                        CELL_SIZE, CELL_SIZE
                    )
                    
                    if grid_offset == 0:  # Linker grid: maze
                        if self.grid[x, y] == 1: color = WALL_COLOR
                        elif self.grid[x, y] == 2: color = HAZARD_COLOR
                        elif self.grid[x, y] == 3: color = GOAL_COLOR
                        else: color = BACKGROUND
                        
                        pygame.draw.rect(self.screen, color, rect)
                        pygame.draw.rect(self.screen, (80, 80, 100), rect, 1)  

                        
                        if x == self.player_pos[0] and y == self.player_pos[1]:
                            pygame.draw.circle(self.screen, PLAYER_COLOR, 
                                             rect.center, CELL_SIZE//4)
                    
                    else:  # Rechter grid: Q-waarden
                        pygame.draw.rect(self.screen, (40, 40, 60), rect)
                        pygame.draw.rect(self.screen, (80, 80, 100), rect, 1)  

                        
                        if self.grid[x, y] != 1:
                            q_values = self.q_table[x, y]
                            max_q = np.max(q_values)
                            best_action = np.argmax(q_values)
                            
                            if max_q > 0:
                                self.draw_arrow(self.screen, rect.center, best_action, max_q / 10)
                                text = self.font.render(f"{max_q:.1f}", True, TEXT_COLOR)
                                self.screen.blit(text, (rect.x + 5, rect.y + 5))
        
        # UI info
        info_y = MARGIN + GRID_SIZE * CELL_SIZE + 10
        info = f"Mode: {self.mode} | Episodes: {self.episodes} | Speed: {self.steps_per_second} steps/sec | Exploration: {self.epsilon*100:.2f}%"
        text = self.font.render(info, True, TEXT_COLOR)
        self.screen.blit(text, (MARGIN, info_y))
        
        controls = "T=Train | M=Manual | R=Reset | +/-=Speed | Arrows=Move"
        controls_text = self.font.render(controls, True, TEXT_COLOR)
        self.screen.blit(controls_text, (MARGIN, info_y + 20))
        
        pygame.display.flip()
    
    def run(self):
        clock = pygame.time.Clock()
        running = True
        
        while running:
            running = self.handle_input()
            
            if self.mode == "training":
                self.training_step()  # Nieuwe batch processing methode
            
            self.draw()
            clock.tick(60)
        
        pygame.quit()

if __name__ == "__main__":
    game = SimpleMaze()
    game.run()


KeyboardInterrupt: 

In [5]:
import pygame
import numpy as np
import random
import os

# Constanten
GRID_SIZE = 6
CELL_SIZE = 125
MARGIN = 30
WINDOW_WIDTH = GRID_SIZE * CELL_SIZE * 2 + MARGIN * 3
WINDOW_HEIGHT = GRID_SIZE * CELL_SIZE + MARGIN * 2 + 80

# Kleuren
BACKGROUND = (30, 30, 50)
WALL_COLOR = (85, 85, 98)
GOAL_COLOR = (50, 180, 80)
HAZARD_COLOR = (220, 80, 60)
TEXT_COLOR = (220, 220, 220)
Q_VALUE_COLOR = (255, 255, 100)

# Maze (0=pad, 1=muur, 2=gevaar, 3=doel)
MAZE = [
    [0, 0, 1, 0, 0, 0],
    [1, 0, 1, 0, 2, 0],
    [0, 0, 0, 0, 1, 3],
    [0, 1, 1, 0, 1, 0],
    [0, 2, 0, 0, 0, 0],
    [0, 1, 0, 1, 0, 0]
]

DIRECTIONS = ['Up', 'Right', 'Down', 'Left']

class PlayerSprite:
    def __init__(self, sprite_folder, cell_size):
        self.images = {}
        for dir_name in DIRECTIONS:
            path = os.path.join(sprite_folder, f"{dir_name}.png")
            img = pygame.image.load(path).convert_alpha()
            self.images[dir_name] = pygame.transform.smoothscale(img, (cell_size-10, cell_size-10))
        self.direction = 'Down'

    def set_direction(self, action):
        self.direction = DIRECTIONS[action]

    def draw(self, surface, center):
        img = self.images[self.direction]
        rect = img.get_rect(center=center)
        surface.blit(img, rect)

class TileSprites:
    def __init__(self, sprite_folder, cell_size):
        finish_path = os.path.join(sprite_folder, "Finish.png")
        hazard_path = os.path.join(sprite_folder, "Hazard.png")
        self.finish = pygame.image.load(finish_path).convert_alpha()
        self.finish = pygame.transform.smoothscale(self.finish, (cell_size-8, cell_size-8))
        self.hazard = pygame.image.load(hazard_path).convert_alpha()
        self.hazard = pygame.transform.smoothscale(self.hazard, (cell_size-8, cell_size-8))

class SimpleMaze:
    def __init__(self):
        pygame.init()
        self.screen = pygame.display.set_mode((WINDOW_WIDTH, WINDOW_HEIGHT))
        pygame.display.set_caption("Q-Learning Maze")
        self.font = pygame.font.SysFont('Arial', 10)  # Kleinere font voor 4 waarden
        
        self.grid = np.array(MAZE)
        self.player_pos = [0, 0]
        self.mode = "manual"
        self.episodes = 0

        try:
            sprite_folder = os.path.join(os.path.dirname(__file__), 'Sprites')
        except NameError:
            sprite_folder = os.path.join(os.getcwd(), 'Sprites')
            
        if not os.path.exists(sprite_folder):
            print(f"Sprites map niet gevonden op: {sprite_folder}")
            print("Zorg dat de Sprites map in dezelfde directory staat als je notebook")

        self.player_sprite = PlayerSprite(sprite_folder, CELL_SIZE)
        self.tile_sprites = TileSprites(sprite_folder, CELL_SIZE)
        self.last_action = 2  # DOWN
        
        # Q-learning
        self.q_table = np.zeros((GRID_SIZE, GRID_SIZE, 4))
        self.epsilon = 0.9
        self.learning_rate = 0.1
        self.discount = 0.9
        
        # Training control 
        self.steps_per_second = 100
        self.min_speed = 10
        self.max_speed = 1000
        self.last_step_time = 0
        self.current_state = None
        self.episode_done = True
        
    def reset(self):
        self.player_pos = [0, 0]
        self.last_action = 2
        self.player_sprite.set_direction(self.last_action)
        return tuple(self.player_pos)
    
    def move(self, action):
        x, y = self.player_pos
        moves = [(-1, 0), (0, 1), (1, 0), (0, -1)]
        dx, dy = moves[action]
        new_x, new_y = x + dx, y + dy
        
        if (0 <= new_x < GRID_SIZE and 0 <= new_y < GRID_SIZE and 
            self.grid[new_x, new_y] != 1):
            self.player_pos = [new_x, new_y]
            self.last_action = action
            self.player_sprite.set_direction(action)
        
        cell = self.grid[self.player_pos[0], self.player_pos[1]]
        if cell == 3: return 100, True
        elif cell == 2: return -50, True
        else: return -1, False
    
    def choose_action(self, state):
        if random.random() < self.epsilon:
            return random.randint(0, 3)
        x, y = state
        return np.argmax(self.q_table[x, y])
    
    def update_q(self, state, action, reward, next_state):
        x, y = state
        next_x, next_y = next_state
        best_next = np.max(self.q_table[next_x, next_y])
        
        self.q_table[x, y, action] += self.learning_rate * (
            reward + self.discount * best_next - self.q_table[x, y, action]
        )
    
    def training_step(self):
        current_time = pygame.time.get_ticks()
        
        if self.episode_done:
            self.current_state = self.reset()
            self.episode_done = False
            self.last_step_time = current_time
            return
        
        delay_ms = 1000 / self.steps_per_second
        elapsed = current_time - self.last_step_time
        steps_to_take = min(50, int(elapsed / delay_ms))
        
        if steps_to_take > 0:
            for _ in range(steps_to_take):
                action = self.choose_action(self.current_state)
                reward, done = self.move(action)
                next_state = tuple(self.player_pos)
                
                self.update_q(self.current_state, action, reward, next_state)
                self.current_state = next_state
                
                if done:
                    self.episode_done = True
                    self.episodes += 1
                    self.epsilon = max(0.1, self.epsilon * 0.999)
                    break
            
            self.last_step_time = current_time
    
    def draw_q_values(self, surface, rect, q_values):
        """Tekent alle 4 Q-waarden in de juiste posities binnen een tile"""
        # Posities: Up, Right, Down, Left
        positions = [
            (rect.centerx, rect.y + 8),      # Up - boven in het midden
            (rect.right - 25, rect.centery), # Right - rechts in het midden  
            (rect.centerx, rect.bottom - 15), # Down - beneden in het midden
            (rect.x + 8, rect.centery)       # Left - links in het midden
        ]
        
        # Kleur bepalen op basis van waarde (groen voor positief, rood voor negatief)
        for i, (q_value, pos) in enumerate(zip(q_values, positions)):
            if abs(q_value) > 0.01:  # Alleen tonen als waarde significant is
                # Kleur bepalen
                if q_value > 0:
                    color = (100, 255, 100)  # Groen voor positieve waarden
                elif q_value < 0:
                    color = (255, 100, 100)  # Rood voor negatieve waarden
                else:
                    color = Q_VALUE_COLOR     # Geel voor neutrale waarden
                
                # Tekst renderen
                text = self.font.render(f"{q_value:.1f}", True, color)
                text_rect = text.get_rect()
                
                # Centreren op de positie
                text_rect.center = pos
                
                # Zorgen dat tekst binnen de tile blijft
                text_rect.clamp_ip(rect.inflate(-4, -4))
                
                surface.blit(text, text_rect)
    
    def handle_input(self):
        for event in pygame.event.get():
            if event.type == pygame.QUIT:
                return False
            
            if event.type == pygame.KEYDOWN:
                if event.key == pygame.K_t:
                    self.mode = "training"
                    self.episode_done = True
                elif event.key == pygame.K_m:
                    self.mode = "manual"
                    self.reset()
                elif event.key == pygame.K_r:
                    self.reset()
                    self.q_table = np.zeros((GRID_SIZE, GRID_SIZE, 4))
                    self.epsilon = 0.9
                    self.episodes = 0
                    self.episode_done = True
                elif event.key == pygame.K_PLUS or event.key == pygame.K_EQUALS:
                    self.steps_per_second = min(self.max_speed, int(self.steps_per_second * 1.25))
                elif event.key == pygame.K_MINUS:
                    self.steps_per_second = max(self.min_speed, int(self.steps_per_second * 0.8))
                
                if self.mode == "manual":
                    moves = {pygame.K_UP: 0, pygame.K_RIGHT: 1, 
                            pygame.K_DOWN: 2, pygame.K_LEFT: 3}
                    if event.key in moves:
                        _, done = self.move(moves[event.key])
                        if done: self.reset()
        return True
    
    def draw(self):
        self.screen.fill(BACKGROUND)
        
        # Beide grids tekenen
        for grid_offset in [0, GRID_SIZE * CELL_SIZE + MARGIN]:
            for x in range(GRID_SIZE):
                for y in range(GRID_SIZE):
                    rect = pygame.Rect(
                        MARGIN + y * CELL_SIZE + grid_offset,
                        MARGIN + x * CELL_SIZE,
                        CELL_SIZE, CELL_SIZE
                    )
                    
                    if grid_offset == 0:  # Linker grid: maze
                        # Eerst de juiste achtergrondkleur tekenen
                        if self.grid[x, y] == 1:
                            pygame.draw.rect(self.screen, WALL_COLOR, rect)
                        elif self.grid[x, y] == 2:
                            pygame.draw.rect(self.screen, HAZARD_COLOR, rect)  
                            self.screen.blit(self.tile_sprites.hazard, rect.inflate(-8, -8))
                        elif self.grid[x, y] == 3:
                            pygame.draw.rect(self.screen, GOAL_COLOR, rect)  
                            self.screen.blit(self.tile_sprites.finish, rect.inflate(-8, -8))
                        else:
                            pygame.draw.rect(self.screen, BACKGROUND, rect)
                        
                        pygame.draw.rect(self.screen, (110, 110, 130), rect, 1)  

                        # Player tekenen als sprite
                        if x == self.player_pos[0] and y == self.player_pos[1]:
                            self.player_sprite.draw(self.screen, rect.center)
                    
                    else:  # Rechter grid: Q-waarden
                        pygame.draw.rect(self.screen, (40, 40, 60), rect)
                        pygame.draw.rect(self.screen, (110, 110, 130), rect, 1)   

                        if self.grid[x, y] != 1:  # Geen Q-waarden voor muren
                            q_values = self.q_table[x, y]
                            self.draw_q_values(self.screen, rect, q_values)
        
        # UI info
        info_y = MARGIN + GRID_SIZE * CELL_SIZE + 10
        info = f"Mode: {self.mode} | Episodes: {self.episodes} | Speed: {self.steps_per_second} steps/sec | Exploration: {self.epsilon*100:.2f}%"
        text = self.font.render(info, True, TEXT_COLOR)
        self.screen.blit(text, (MARGIN, info_y))
        
        controls = "T=Train | M=Manual | R=Reset | +/-=Speed | Arrows=Move"
        controls_text = self.font.render(controls, True, TEXT_COLOR)
        self.screen.blit(controls_text, (MARGIN, info_y + 20))
        
        pygame.display.flip()
    
    def run(self):
        clock = pygame.time.Clock()
        running = True
        
        while running:
            running = self.handle_input()
            
            if self.mode == "training":
                self.training_step()
            
            self.draw()
            clock.tick(60)
        
        pygame.quit()

if __name__ == "__main__":
    game = SimpleMaze()
    game.run()


# Laatste versie

In [10]:
import pygame
import numpy as np
import random
import os

# Constanten
GRID_SIZE = 6
CELL_SIZE = 125
MARGIN = 30
WINDOW_WIDTH = GRID_SIZE * CELL_SIZE * 2 + MARGIN * 3
WINDOW_HEIGHT = GRID_SIZE * CELL_SIZE + MARGIN * 2 + 80

# Kleuren
BACKGROUND = (30, 30, 50)
WALL_COLOR = (85, 85, 98)
GOAL_COLOR = (50, 180, 80)
HAZARD_COLOR = (220, 80, 60)
TEXT_COLOR = (220, 220, 220)
Q_VALUE_COLOR = (255, 255, 100)

# Maze layout
MAZE = [
    [0, 0, 1, 0, 0, 0],
    [1, 0, 1, 0, 2, 0],
    [0, 0, 0, 0, 1, 3],
    [0, 1, 1, 0, 1, 0],
    [0, 2, 0, 0, 0, 0],
    [0, 1, 0, 1, 0, 0]
]

DIRECTIONS = ['Up', 'Right', 'Down', 'Left']

class PlayerSprite:
    # Laadt en beheert player sprites voor verschillende richtingen
    def __init__(self, sprite_folder, cell_size):
        self.images = {}
        for dir_name in DIRECTIONS:
            path = os.path.join(sprite_folder, f"{dir_name}.png")
            img = pygame.image.load(path).convert_alpha()
            self.images[dir_name] = pygame.transform.smoothscale(img, (cell_size-10, cell_size-10))
        self.direction = 'Down'

    # Stelt de richting van de sprite in
    def set_direction(self, action):
        self.direction = DIRECTIONS[action]

    # Tekent de sprite op het scherm
    def draw(self, surface, center):
        img = self.images[self.direction]
        rect = img.get_rect(center=center)
        surface.blit(img, rect)

class TileSprites:
    # Laadt en beheert sprites voor speciale tiles
    def __init__(self, sprite_folder, cell_size):
        finish_path = os.path.join(sprite_folder, "Finish2.png")
        hazard_path = os.path.join(sprite_folder, "Hazard.png")
        self.finish = pygame.image.load(finish_path).convert_alpha()
        self.finish = pygame.transform.smoothscale(self.finish, (cell_size-8, cell_size-8))
        self.hazard = pygame.image.load(hazard_path).convert_alpha()
        self.hazard = pygame.transform.smoothscale(self.hazard, (cell_size-8, cell_size-8))

class SimpleMaze:
    # Initialiseert het spel en alle componenten
    def __init__(self):
        pygame.init()
        self.screen = pygame.display.set_mode((WINDOW_WIDTH, WINDOW_HEIGHT))
        pygame.display.set_caption("Q-Learning Maze")
        self.font = pygame.font.SysFont('Arial', 20)
        
        self.grid = np.array(MAZE)
        self.player_pos = [0, 0]
        self.mode = "manual"
        self.episodes = 0
        self.wins = 0 
        try:
            sprite_folder = os.path.join(os.path.dirname(__file__), 'Sprites')
        except NameError:
            sprite_folder = os.path.join(os.getcwd(), 'Sprites')
            
        if not os.path.exists(sprite_folder):
            print(f"Sprites map niet gevonden op: {sprite_folder}")
            print("Zorg dat de Sprites map in dezelfde directory staat als je notebook")
            
        self.player_sprite = PlayerSprite(sprite_folder, CELL_SIZE)
        self.tile_sprites = TileSprites(sprite_folder, CELL_SIZE)
        self.last_action = 2
        
        self.q_table = np.zeros((GRID_SIZE, GRID_SIZE, 4))
        self.epsilon = 0.9
        self.learning_rate = 0.1
        self.discount = 0.9
        
        self.steps_per_second = 100
        self.min_speed = 10
        self.max_speed = 1000
        self.last_step_time = 0
        self.current_state = None
        self.episode_done = True
        
    # Reset de speler naar de startpositie
    def reset(self):
        self.player_pos = [0, 0]
        self.last_action = 2
        self.player_sprite.set_direction(self.last_action)
        return tuple(self.player_pos)
    
    # Verwerkt een beweging en geeft reward en done status terug
    def move(self, action):
        x, y = self.player_pos
        moves = [(-1, 0), (0, 1), (1, 0), (0, -1)]
        dx, dy = moves[action]
        new_x, new_y = x + dx, y + dy
        
        if (0 <= new_x < GRID_SIZE and 0 <= new_y < GRID_SIZE and 
            self.grid[new_x, new_y] != 1):
            self.player_pos = [new_x, new_y]
            self.last_action = action
            self.player_sprite.set_direction(action)
        
        cell = self.grid[self.player_pos[0], self.player_pos[1]]
        if cell == 3: return 100, True
        elif cell == 2: return -50, True
        else: return -1, False
    
    # Kiest een actie met epsilon-greedy strategie
    def choose_action(self, state):
        if random.random() < self.epsilon:
            return random.randint(0, 3)
        x, y = state
        return np.argmax(self.q_table[x, y])
    
    # Update de Q-waarden met Q-learning algoritme
    def update_q(self, state, action, reward, next_state):
        x, y = state
        next_x, next_y = next_state
        best_next = np.max(self.q_table[next_x, next_y])
        
        self.q_table[x, y, action] += self.learning_rate * (
            reward + self.discount * best_next - self.q_table[x, y, action]
        )
    
    # Voert training stappen uit op basis van snelheid instelling
    def training_step(self):
        current_time = pygame.time.get_ticks()
        
        if self.episode_done:
            self.current_state = self.reset()
            self.episode_done = False
            self.last_step_time = current_time
            return
        
        delay_ms = 1000 / self.steps_per_second
        elapsed = current_time - self.last_step_time
        steps_to_take = min(50, int(elapsed / delay_ms))
        
        if steps_to_take > 0:
            for _ in range(steps_to_take):
                action = self.choose_action(self.current_state)
                reward, done = self.move(action)
                next_state = tuple(self.player_pos)
                
                self.update_q(self.current_state, action, reward, next_state)
                self.current_state = next_state
                
                if done:
                    self.episode_done = True
                    self.episodes += 1
                    if reward == 100:  
                        self.wins += 1
                    self.epsilon = max(0.1, self.epsilon * 0.999)
                    break
            
            self.last_step_time = current_time
    
    # Tekent Q-waarden in een grid cel met kleurcodering
    def draw_q_values(self, surface, rect, q_values):
        positions = [
            (rect.centerx, rect.y + 8),
            (rect.right - 25, rect.centery),
            (rect.centerx, rect.bottom - 15),
            (rect.x + 8, rect.centery)
        ]
        
        max_q_index = np.argmax(q_values)
        
        for i, (q_value, pos) in enumerate(zip(q_values, positions)):
            if abs(q_value) > 0.01:
                if i == max_q_index:
                    color = (255, 255, 255)
                elif q_value > 0:
                    color = (100, 255, 100)
                elif q_value < 0:
                    color = (255, 100, 100)
                else:
                    color = Q_VALUE_COLOR
                
                text = self.font.render(f"{q_value:.1f}", True, color)
                text_rect = text.get_rect()
                text_rect.center = pos
                text_rect.clamp_ip(rect.inflate(-4, -4))
                surface.blit(text, text_rect)
    
    # Verwerkt toetsenbord input en game controls
    def handle_input(self):
        for event in pygame.event.get():
            if event.type == pygame.QUIT:
                return False
            
            if event.type == pygame.KEYDOWN:
                if event.key == pygame.K_t:
                    self.mode = "training"
                    self.episode_done = True
                elif event.key == pygame.K_m:
                    self.mode = "manual"
                    self.reset()
                elif event.key == pygame.K_r:
                    self.reset()
                    self.q_table = np.zeros((GRID_SIZE, GRID_SIZE, 4))
                    self.epsilon = 0.9
                    self.episodes = 0
                    self.wins = 0  
                    self.episode_done = True
                elif event.key == pygame.K_PLUS or event.key == pygame.K_EQUALS:
                    self.steps_per_second = min(self.max_speed, int(self.steps_per_second * 1.25))
                elif event.key == pygame.K_MINUS:
                    self.steps_per_second = max(self.min_speed, int(self.steps_per_second * 0.8))
                
                if self.mode == "manual":
                    moves = {pygame.K_UP: 0, pygame.K_RIGHT: 1, 
                            pygame.K_DOWN: 2, pygame.K_LEFT: 3}
                    if event.key in moves:
                        _, done = self.move(moves[event.key])
                        if done: self.reset()
        return True
    
    # Tekent het complete spel scherm met maze en Q-waarden
    def draw(self):
        self.screen.fill(BACKGROUND)
        
        for grid_offset in [0, GRID_SIZE * CELL_SIZE + MARGIN]:
            for x in range(GRID_SIZE):
                for y in range(GRID_SIZE):
                    rect = pygame.Rect(
                        MARGIN + y * CELL_SIZE + grid_offset,
                        MARGIN + x * CELL_SIZE,
                        CELL_SIZE, CELL_SIZE
                    )
                    
                    if grid_offset == 0:
                        if self.grid[x, y] == 1:
                            pygame.draw.rect(self.screen, WALL_COLOR, rect)
                        elif self.grid[x, y] == 2:
                            pygame.draw.rect(self.screen, HAZARD_COLOR, rect)  
                            self.screen.blit(self.tile_sprites.hazard, rect.inflate(-8, -8))
                        elif self.grid[x, y] == 3:
                            pygame.draw.rect(self.screen, GOAL_COLOR, rect)  
                            self.screen.blit(self.tile_sprites.finish, rect.inflate(-8, -8))
                        else:
                            pygame.draw.rect(self.screen, BACKGROUND, rect)
                        
                        pygame.draw.rect(self.screen, (110, 110, 130), rect, 1)
                        
                        if x == self.player_pos[0] and y == self.player_pos[1]:
                            self.player_sprite.draw(self.screen, rect.center)
                    
                    else:
                        pygame.draw.rect(self.screen, (40, 40, 60), rect)
                        pygame.draw.rect(self.screen, (110, 110, 130), rect, 1)
                        if self.grid[x, y] != 1:
                            q_values = self.q_table[x, y]
                            self.draw_q_values(self.screen, rect, q_values)
        
        info_y = MARGIN + GRID_SIZE * CELL_SIZE + 10
        winrate = (self.wins / self.episodes) * 100 if self.episodes > 0 else 0  # Toegevoegd voor winrate
        info = f"Mode: {self.mode} | Episodes: {self.episodes} | Wins: {self.wins} | Winrate: {winrate:.2f}% | Speed: {self.steps_per_second} steps/sec | Exploration: {self.epsilon*100:.2f}%"  # Toegevoegd voor winrate
        text = self.font.render(info, True, TEXT_COLOR)
        self.screen.blit(text, (MARGIN, info_y))
        
        controls = "T=Train | M=Manual | R=Reset | +/-=Speed | Arrows=Move"
        controls_text = self.font.render(controls, True, TEXT_COLOR)
        self.screen.blit(controls_text, (MARGIN, info_y + 20))
        
        pygame.display.flip()
    
    # Hoofdgame loop
    def run(self):
        clock = pygame.time.Clock()
        running = True
        
        while running:
            running = self.handle_input()
            
            if self.mode == "training":
                self.training_step()
            
            self.draw()
            clock.tick(60)
        
        pygame.quit()

if __name__ == "__main__":
    game = SimpleMaze()
    game.run()
